Answer the following questions at the corresponding question numbers __in the "Assignment 1 test" SunLearn quiz__, doing calculations in this notebook where necessary/desired.  __Give answers rounded to 4 decimal places - answers must be exactly correct to receive marks.__

__Remember to divide by N, not N-1, in the computation of all variances in this test__.


In [1]:
import numpy as np

**Question 1 (3 marks).** Consider the 4x2 matrix $M$ below. // __*Vraag 1 (3 punte).*__ *Beskou die 4x2 matriks $M$ hieronder.*

In [8]:
M = np.array([[6, 12, 24], [2, 6, 10], [-1, -2, -4], [0, 0, 0]])
M

array([[ 6, 12, 24],
       [ 2,  6, 10],
       [-1, -2, -4],
       [ 0,  0,  0]])

(a) Give the third largest singular value of $M$. [1]



In [7]:
U, S, V = np.linalg.svd(M)
print(f"Third largest singular value of M: {S[-1]}")

Third largest singular value of M: 1.8680885607560182e-16


(b) Viewing $M$ as a sample of *four 3-dimensional data points*, give (i) the sample covariance between the first and the third feature [1] and (ii) the eigenvalue corresponding to the principal eigenvector of the sample covariance matrix [1].

In [13]:
N = 4
D = M - np.mean(M, axis=0)
#sample covariance matrix
S = (D.T @ D) / N
S

array([[  7.1875,  14.5   ,  28.875 ],
       [ 14.5   ,  30.    ,  59.    ],
       [ 28.875 ,  59.    , 116.75  ]])

In [17]:
e_vals, e_vecs = np.linalg.eigh(S)
print(f"Eigen values:\n{e_vals}")
print(f"Eigen vectors:\n{e_vecs}")

Eigen values:
[-1.97028785e-14  2.09786507e-01  1.53727713e+02]
Eigen vectors:
[[-0.81649658  0.535696   -0.21532099]
 [-0.40824829 -0.79940928 -0.44076994]
 [ 0.40824829  0.27198273 -0.87141192]]


**Question 2 (4 marks).** The code below should load the data matrix from the Breast Cancer Wisconsin benchmark data set into the variable `br`.

In [18]:
from sklearn.datasets import load_breast_cancer
br = load_breast_cancer().data
print(br.shape)

(569, 30)


(a) The input features of this data set are all positive, and on various scales.  Give the standard deviation of the feature with the largest variance in this data set. [1]

In [24]:
stds = []
for col in br.T:
    stds.append(np.std(col))
print(max(stds))

568.8564589532671


In [25]:
stds = np.std(br, axis=0)
print(max(stds))

568.8564589532672


(b) Centre and scale the data so that the variance of each feature is one.  Give the smallest value of the second feature of the transformed data. [1]

In [42]:
D = br - np.mean(br, axis=0)
br_new = D / stds
print(min(br_new[:, 1]))

-2.229248512170399


(c) How many PCA features are *needed* to retain at least 95% of the variance in this (standardised) data set? [1]

In [48]:
pca = tPCA()
pca.fit_svd(br_new.T)
var_ratios = pca.explained_variance_ratio

count = 0
explained_var = 0
for i in var_ratios:
    count += 1
    explained_var += i
    
    if explained_var >= 0.95:
        print(count)
        print(float(explained_var))
        break

10
0.9515688143366668


(d) What is the score for the first feature associated with the hundredth point in the data set when using this number of principal components? [1]

In [52]:
pca = tPCA(n_components=10)
Z = pca.fit_svd_transform(br_new.T)
Z[0, 99]

np.float64(-0.6162808698250983)

**Question 3 (3 marks).** Suppose you receive a data set containing vector representations summarizing short social media posts, where the vectors are 512-dimensional.  Each vector is associated with one of 12 topics.  Viewing these topics as class labels, assume you decide to process this data using linear discriminant analysis for dimensionality reduction.

(a) How many rows will the within-class scatter matrix you need to compute in this process have? [1]

(b) What is the maximum rank of the between-class scatter matrix you need to compute in this process? [1]

(c) What is the maximum number of ("non-empty") dimensions your transformed data may have when you have completed this pre-processing? [1]


In [ ]:
# d = 512, N = ?, k = 12

# within class scatter = dxd, 512x512

In [ ]:
#Rank(S_b) = k-1 = 11

In [ ]:
#LDA can produce a maximum of min(d, k-1) dimensions, = 11

**Question 4 (2 marks).** Consider applying linear discriminant analysis to the (full) Fisher Iris data set, loaded below, __without initially centering or scaling the data__.

(a) Give the bottom-right entry of the total within-class scatter matrix. (Use the normalized form as in Formula (3.10) in the course notes.) [1]

(b) Give the proportion of within-class variance explained by the fourth principal component used in the whitening transform. [1]

In [80]:
from sklearn.datasets import load_wine
wine_data = load_wine()
W_X = wine_data.data # observations
W_y = wine_data.target # labels
W_X.shape

(178, 13)

In [81]:
S_w, S_b = compute_normalized_scatter_matrices(W_X.T, W_y)
S_w[-1, -1]

np.float64(29206.99060303627)

In [82]:
lda = LDA(normalise=True)
lda.fit(W_X.T, W_y)
print(lda.explained_variance_ratio)

[0.68747889 0.31252111]


In [65]:
def _validate_xy(X, y):
    """Validate X/y shape alignment for CS315 convention.

    Parameters:
    -------------------
    X : np.ndarray
        Data matrix with shape ``(n_features, n_samples)``.
    y : np.ndarray
        Label vector with shape ``(n_samples,)``.

    Returns:
    --------------
    tuple[np.ndarray, np.ndarray]
        Sanitized arrays ``(X, y)``.
    """
    X = np.asarray(X)
    y = np.asarray(y)

    if X.ndim != 2:
        raise ValueError(f"Expected X to be 2D, got {X.ndim}D.")
    if y.ndim != 1:
        raise ValueError(f"Expected y to be 1D, got {y.ndim}D.")
    if X.shape[1] != y.shape[0]:
        raise ValueError(
            f"X/y mismatch: X has {X.shape[1]} samples, y has {y.shape[0]} labels."
        )
    return X, y


def compute_class_means(X, y):
    """Compute per-class mean vectors.

    Parameters:
    -------------------
    X : np.ndarray
        Data matrix with shape ``(n_features, n_samples)``.
    y : np.ndarray
        Label vector with shape ``(n_samples,)``.

    Returns:
    --------------
    tuple[np.ndarray, np.ndarray]
        ``(class_labels, class_means)`` where:
        - ``class_labels`` has shape ``(n_classes,)``
        - ``class_means`` has shape ``(n_features, n_classes)``
    """
    X, y = _validate_xy(X, y)
    class_labels = np.unique(y)
    means = []
    for c in class_labels:
        means.append(np.mean(X[:, y == c], axis=1))
    return class_labels, np.column_stack(means)


def compute_scatter_matrices(X, y):
    """Compute within-class and between-class scatter matrices.

    Parameters:
    -------------------
    X : np.ndarray
        Data matrix with shape ``(n_features, n_samples)``.
    y : np.ndarray
        Label vector with shape ``(n_samples,)``.

    Returns:
    --------------
    tuple[np.ndarray, np.ndarray]
        ``(S_w, S_b)`` each with shape ``(n_features, n_features)``.
    """
    X, y = _validate_xy(X, y)
    class_labels, class_means = compute_class_means(X, y)
    global_mean = np.mean(X, axis=1)

    d = X.shape[0]
    S_w = np.zeros((d, d))
    S_b = np.zeros((d, d))

    for i, c in enumerate(class_labels):
        X_c = X[:, y == c]
        m_c = class_means[:, i]
        D_c = X_c - m_c[:, None]
        S_w += D_c @ D_c.T
        dm = m_c - global_mean
        S_b += X_c.shape[1] * np.outer(dm, dm)

    return S_w, S_b

def compute_normalized_scatter_matrices(X, y, mode="pooled_mle"):
    """Compute normalized within/between-class scatter matrices.

    Different libraries use different normalization conventions.
    This helper makes those conventions explicit and easy to compare.

    Parameters:
    -------------------
    X : np.ndarray
        Data matrix with shape ``(n_features, n_samples)``.
    y : np.ndarray
        Label vector with shape ``(n_samples,)``.
    mode : str
        Normalization strategy for ``S_w``:
        - ``"pooled_unbiased"``: ``S_w / (n_samples - n_classes)``
        - ``"pooled_mle"``: ``S_w / n_samples``
        - ``"weighted_class_cov"``: ``sum_k pi_k * cov_k_unbiased``
          where ``pi_k = n_k / n_samples``.

    Returns:
    --------------
    tuple[np.ndarray, np.ndarray]
        ``(S_w_norm, S_b_norm)`` each with shape ``(n_features, n_features)``.
        ``S_b_norm`` is always returned as ``S_b / n_samples``.
    """
    X, y = _validate_xy(X, y)
    S_w, S_b = compute_scatter_matrices(X, y)

    n_samples = X.shape[1]
    class_labels = np.unique(y)
    n_classes = class_labels.size

    if mode == "pooled_unbiased":
        denom = n_samples - n_classes
        if denom <= 0:
            raise ValueError("Need n_samples > n_classes for pooled_unbiased mode.")
        S_w_norm = S_w / denom
    elif mode == "pooled_mle":
        if n_samples <= 0:
            raise ValueError("n_samples must be positive.")
        S_w_norm = S_w / n_samples
    elif mode == "weighted_class_cov":
        d = X.shape[0]
        S_w_norm = np.zeros((d, d))
        for c in class_labels:
            X_c = X[:, y == c]
            n_k = X_c.shape[1]
            if n_k < 2:
                raise ValueError(
                    "Each class must have at least 2 samples in weighted_class_cov mode."
                )
            D_c = X_c - np.mean(X_c, axis=1, keepdims=True)
            cov_k_unbiased = (D_c @ D_c.T) / (n_k - 1)
            S_w_norm += (n_k / n_samples) * cov_k_unbiased
    else:
        raise ValueError(
            "Unknown mode. Use one of: "
            "'pooled_unbiased', 'pooled_mle', 'weighted_class_cov'."
        )

    S_b_norm = S_b / n_samples
    return S_w_norm, S_b_norm

In [44]:
class tPCA:
    """
    Principal Component Analysis (PCA) for matrices with shape (d, N).
    
    This implementation follows the CS315 module convention:
    - rows are features/variables (dimension d)
    - columns are observations/samples (count N)
    
    The class supports two fitting routes:
    - `fit_svd`: PCA via singular value decomposition (SVD) of centred data
    - `fit_eigh`: PCA via eigendecomposition of the covariance matrix
    
    Stored attributes after fitting:
    - `x_bar` : feature-wise training mean vector, shape (d,)
    - `principle_components` : matrix whose columns are retained principal directions, shape (d, k)
    - `explained_variance` : eigenvalues for retained components, shape (k,)
    - `explained_variance_ratio` : retained variance fractions, shape (k,)
    """
    
    def __init__(self, n_components=None, whiten=False, ddof=None):
        """
        Initialise PCA hyperparameters and fitted-state placeholders.
        
        Parameters:
        -----------
        n_components : int or None, default=None
            Number of principal components to retain.
            If None, uses the maximum valid value for the chosen fit method.
        whiten : bool, default=False
            If True, transformed coordinates are scaled by inverse standard
            deviation of each retained component.
        ddof : int or None, default=None
            Delta degrees of freedom used in covariance/eigenvalue scaling.
            If None, defaults to 0.
        """
        self.n_components = n_components
        self.whiten = whiten
        self.ddof = 0 if ddof is None else ddof
        
        # Mean vector of features calculated on 'training' data (used for centering)
        self.x_bar = None
        # Columns are principal directions (eigenvectors of covariance matrix)
        self.principle_components = None
        # Eigenvalues of covariance matrix (variance captured per principal component)
        self.explained_variance = None
        # Fraction of total variance explained by each principal component
        self.explained_variance_ratio = None
        
    #===========================================================================
    # External API
    #===========================================================================

    def fit_svd(self, X):
        """
        Perform SVD to derive the principal components and singular values.
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            The input matrix where `d` represents the feature dimension 
            and `N` is the number of observations.
            
        Returns:
        --------
        None
        """
        self._set_n_components(X, method="svd")
        
        # Perform Singular Value Decomposition
        D = self._centre_train(X)
        U, S, V = np.linalg.svd(D, full_matrices=False)
        
        self.principle_components = U[:, :self.n_components]
        singular_values = S[:self.n_components]
        # compute total varaince based on all singular values
        total_variance = np.sum(self._expl_var(S, D.shape[1]))
        self.explained_variance = self._expl_var(singular_values, D.shape[1])
        self._set_expl_var_ratios(total_variance)
        

    def fit_eigh(self, X):
        """
        Perform covariance-based PCA using eigendecomposition.
        
        This method:
        - centres training data
        - builds covariance matrix S = (1/(N-ddof)) * D D^T
        - eigendecomposes S
        - sorts eigenpairs in descending-variance order
        - stores retained principal directions and explained variance data
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            The input matrix where `d` represents the feature dimension
            and `N` is the number of observations.
        
        Returns:
        --------
        None
        """
        self._set_n_components(X, method="eigh")
            
        D = self._centre_train(X)
        S = self._build_cov(D)
        eigenvalues, eigenvectors = np.linalg.eigh(S)
        #sort eigenvalues and eigenvectors in non-increasing order
        eigenvalues = np.flip(eigenvalues)
        eigenvectors = np.fliplr(eigenvectors)
        
        self.principle_components = eigenvectors[:, :self.n_components]
        self.explained_variance = eigenvalues[:self.n_components]
        total_variance = sum(eigenvalues)
        self._set_expl_var_ratios(total_variance)
            
    
    def transform(self, X_new):
        """
        Project data into the fitted principal-component coordinate system.
        
        This method first centres `X_new` with the training mean `x_bar`, then
        projects onto retained principal directions. If whitening is enabled,
        each component score is additionally divided by sqrt(eigenvalue) so
        each retained component has unit variance.
        
        Parameters:
        -----------
        X_new : ndarray of shape (d, M)
            Data to project, where rows are features and columns are
            observations.
        
        Returns:
        --------
        Z : ndarray of shape (k, M)
            Projected coordinates (component scores) in retained PC basis.
        """
        if self.principle_components is None:
            raise Exception("Please fit the PCA First")
        
        D_new = self._centre_oos(X_new)
        if self.whiten:
            if np.any(self.explained_variance <= 0):
                raise ValueError(
                    "Whitening requires strictly positive explained variance "
                    "for all retained components. Reduce n_components or disable whiten."
                )
            A = np.diag(np.ones(shape=len(self.explained_variance)) / np.sqrt(self.explained_variance))
            Z = A @ self.principle_components.T @ D_new
        else: 
            Z = self.principle_components.T @ D_new
        
        return Z
    
    
    def fit_svd_transform(self, X):
        """
        Convenience wrapper equivalent to:
        1) `fit_svd(X)`
        2) `transform(X)`
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            Training data matrix with rows as features and columns as
            observations.
        
        Returns:
        --------
        Z : ndarray of shape (k, N)
            Projected training data in retained PC basis.
        """
        self.fit_svd(X)
        return self.transform(X)
    
    
    def fit_eigh_transform(self, X):
        """
        Convenience wrapper equivalent to:
        1) `fit_eigh(X)`
        2) `transform(X)`
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            Training data matrix with rows as features and columns as
            observations.
        
        Returns:
        --------
        Z : ndarray of shape (k, N)
            Projected training data in retained PC basis.
        """
        self.fit_eigh(X)
        return self.transform(X)
    
    
    def inverse_transform(self, Z):
        """
        Map component scores back to the original feature space.
        
        If whitening was used during `transform`, this method first reverses
        the whitening scale and then reconstructs centred data with the
        retained basis before adding back the training mean vector.
        
        Parameters:
        -----------
        Z : ndarray of shape (k, M)
            Component scores in retained PC basis.
        
        Returns:
        --------
        X_hat : ndarray of shape (d, M)
            Reconstructed data in the original feature space.
        """
        if self.principle_components is None:
            raise Exception("Please fit the PCA First")
        
        if self.whiten:
            A = np.diag(np.sqrt(self.explained_variance))
            Z = A @ Z
        
        D_hat = self.principle_components @ Z
        x_bar_matrix = np.outer(self.x_bar, np.ones(shape=D_hat.shape[1]).T)
        X_hat = D_hat +  x_bar_matrix
        return X_hat


    #===========================================================================
    # Internal Helpers
    #===========================================================================
    
    def _set_n_components(self, X, method):
        """
        Validate and set the number of principal components.
        
        For SVD with X in shape (d, N), at most min(d, N) singular directions
        can be recovered. For covariance eigendecomposition, the maximum count
        is d.
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            Input data matrix using rows=features and columns=observations.
        method : {"svd", "eigh"}
            Fit method being used, which determines the maximum allowable
            number of retained components.
        
        Returns:
        --------
        None
        """
        d, N = X.shape
        max_components = min(d, N) if method == "svd" else d

        if self.n_components is None:
            self.n_components = max_components
            return

        if not isinstance(self.n_components, (int, np.integer)):
            raise TypeError("n_components must be an integer or None")
        if self.n_components <= 0:
            raise ValueError("n_components must be >= 1")
        if self.n_components > max_components:
            raise ValueError(
                f"n_components={self.n_components} exceeds maximum "
                f"allowed ({max_components}) for method='{method}' and "
                f"input shape {X.shape}"
            )

    
    def _centre_train(self, X):
        """
        Centre the data by subtracting the mean vector x_bar from each column.
        
        This method also sets the x_bar class variable
        
        Parameters:
        -----------
        X : ndarray of shape (d, N)
            The input matrix where `d` represents the feature dimension 
            and `N` is the number of observations.
            
        Returns:
        --------
        D : ndarray of shape (d, N)
            The Centred data matrix of the "training" observations
        """
        self.x_bar = np.mean(X, axis=1)
        x_bar_matrix = np.outer(self.x_bar, np.ones(shape=X.shape[1]).T)
        D = X - x_bar_matrix
        return D
    
    
    def _centre_oos(self, X_new):
        """
        Centre out-of-sample data using the stored training mean vector.
        
        Parameters:
        -----------
        X_new : ndarray of shape (d, M)
            New data matrix where rows are features and columns are
            observations.
        
        Returns:
        --------
        D_new : ndarray of shape (d, M)
            Centred out-of-sample data matrix.
        """
        x_bar_matrix = np.outer(self.x_bar, np.ones(shape=X_new.shape[1]).T)
        D_new = X_new - x_bar_matrix
        return D_new


    def _build_cov(self, D):
        """
        Build covariance matrix from centred data.
        
        Covariance is computed as:
            S = (1/(N-ddof)) * D D^T
        where D has shape (d, N), giving S shape (d, d).
        
        Parameters:
        -----------
        D : ndarray of shape (d, N)
            Centred data matrix.
        
        Returns:
        --------
        S : ndarray of shape (d, d)
            Covariance matrix.
        """
        if D.shape[1] - self.ddof <= 0:
            raise ValueError(
                f"Invalid ddof={self.ddof} for N={D.shape[1]}. "
                "Require N - ddof > 0."
            )
        S = (1 / (D.shape[1] - self.ddof)) * (D @ D.T)
        return S
    
    
    def _expl_var(self, sing_vals, N):
        """
        Calculate explained variance as squared singular values divided by (N-ddof).
        Singular values are related to eigenvalues via:
            lambda_i = (sing_vals)_i^2 / (N - ddof)
        
        Parameters:
        -----------
        sing_vals : np.ndarray
            Singular values corresponding to centred data.
        N : int
            The number of observations in the "training" set
        
        Returns:
        --------
        explained_variance : ndarray of shape (len(sing_vals),)
            The explained variance (eigenvalues) based on the provided singular values.
        """
        if N - self.ddof <= 0:
            raise ValueError(
                f"Invalid ddof={self.ddof} for N={N}. Require N - ddof > 0."
            )
        explained_variance = np.empty(len(sing_vals))
        for i, s in enumerate(sing_vals):
            explained_variance[i] = s**2 / (N - self.ddof)
        
        return explained_variance
        
        
    def _set_expl_var_ratios(self, total_variance):
        """
        Set explained variance ratios as:
            explained_variance[i] / sum(explained_variance)
        
        Parameters:
        -----------
        total_variance : float
            Total variance in the full (non-truncated) eigenspectrum used as
            denominator for ratio computation.
        
        Returns:
        --------
        None, just sets the class variable 'explained_variance_ratio'
        """
        if np.ndim(total_variance) != 0:
            raise ValueError("total_variance must be a scalar.")
        if total_variance <= 0:
            raise ValueError("total_variance must be strictly positive.")

        self.explained_variance_ratio = np.empty(len(self.explained_variance))
        
        for i, lambda_ in enumerate(self.explained_variance):
            self.explained_variance_ratio[i] = lambda_ / total_variance


In [77]:
import numpy as np

class LDA:
    """Linear Discriminant Analysis for dimensionality reduction.

    This implementation follows the convention used in CS315:
        features are rows and observations are columns.
    """

    def __init__(self, n_components=None, normalise=False):
        """Initialize the LDA model.

        Parameters:
        -----------
        n_components : int | None
            Target dimensionality. If None, the maximum valid dimensionality is used.

        Returns:
        --------
        None
        """
        self.n_components = n_components
        self.normalise = normalise

        self.class_labels = None
        self.global_mean = None
        self.class_means = None
        self.transformation_matrix = None
        self.explained_variance_ratio = None
        

    def fit(self, X, y):
        """Fit LDA projection directions from training data.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        LDA
            The fitted model instance.
        """
        X = np.asarray(X)
        y = np.asarray(y)

        # Validate model parameters and input arrays.
        self._validate_input(X, y)

        # Compute means.
        self.global_mean = np.mean(X, axis=1)
        self.class_means = self._compute_class_means(X, y)

        # Compute scatter matrices.
        S_w = self._compute_within_scatter(X, y)
        class_counts = self._compute_class_counts(X, y)
        S_b = self._compute_between_scatter(X.shape[0], class_counts)
        
        if self.normalise:
            S_w = S_w / X.shape[1]
            S_b = S_b / X.shape[1]

        # Compute whitening matrix.
        P = self._compute_whitening(S_w)
        # Whiten S_b.
        S_b_white = P @ S_b @ P.T
        # Compute directions of maximal between-class variance.
        V_k = self._eigendecompose_S_b_white(S_b_white)
        # Compute transformation matrix.
        self.transformation_matrix = P.T @ V_k
        return self
    

    def transform(self, X_new):
        """Project new data into the fitted LDA subspace.

        Parameters:
        -----------
        X_new : np.ndarray
            Matrix of shape ``(n_features, n_new_samples)``.

        Returns:
        --------
        np.ndarray
            Projected matrix of shape ``(n_components, n_new_samples)``.
        """
        X_new = np.asarray(X_new)

        # Validate that LDA has been fitted.
        if self.global_mean is None or self.transformation_matrix is None:
            raise RuntimeError("Call fit before transform.")
        if np.ndim(X_new) != 2:
            raise ValueError(f"LDA expects X_new to be 2D, got {np.ndim(X_new)}D.")
        if X_new.shape[0] != self.global_mean.shape[0]:
            raise ValueError(
                "X_new has incompatible feature dimension: "
                f"expected {self.global_mean.shape[0]}, got {X_new.shape[0]}."
            )

        # Center new data using training global mean.
        D_new = X_new - np.outer(self.global_mean, np.ones(shape=X_new.shape[1]))

        # Project into LDA space.
        Z = self.transformation_matrix.T @ D_new
        return Z


    def fit_transform(self, X, y):
        """Fit the model and return the projected training data.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        np.ndarray
            Projected training matrix of shape ``(n_components, n_samples)``.
        """
        self.fit(X, y)
        return self.transform(X)
    

    def _validate_input(self, X, y):
        """Validate input shapes and model hyperparameters.

        This function also sets class labels for `n_components`
        when the value is not explicitly provided.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        None
        """
        if np.ndim(X) != 2:
            raise ValueError(f"LDA expects X to be 2D, got {np.ndim(X)}D.")
        if np.ndim(y) != 1:
            raise ValueError(f"LDA expects y to be 1D, got {np.ndim(y)}D.")
        if X.shape[1] != y.shape[0]:
            raise ValueError(
                "X and y are misaligned: "
                f"X has {X.shape[1]} samples while y has {y.shape[0]} labels."
            )

        self.class_labels = np.unique(y)
        if len(self.class_labels) < 2:
            raise ValueError("LDA requires at least 2 distinct classes.")

        if self.n_components is not None:
            if not isinstance(self.n_components, (int, np.integer)):
                raise TypeError("n_components must be an integer or None.")
            if self.n_components < 1:
                raise ValueError("n_components must be at least 1.")

        max_components = min(X.shape[0], len(self.class_labels) - 1)
        if self.n_components is None:
            self.n_components = max_components
        elif self.n_components > max_components:
            raise ValueError(
                f"LDA can produce at most {max_components} components, "
                f"got {self.n_components}."
            )


    def _compute_class_means(self, X, y):
        """Compute the mean feature vector for each class.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        np.ndarray
            Class-mean matrix of shape ``(n_features, n_classes)``.
        """
        class_means = []
        for c in self.class_labels:
            mask = (y == c)
            X_c = X[:, mask]
            class_means.append(np.mean(X_c, axis=1))

        return np.column_stack(class_means)


    def _compute_within_scatter(self, X, y):
        """Compute within-class scatter matrix.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        np.ndarray
            Within-class scatter matrix of shape ``(n_features, n_features)``.
        """
        S_w = np.zeros((X.shape[0], X.shape[0]))
        for i, c in enumerate(self.class_labels):
            mask = (y == c)
            X_c = X[:, mask]
            D_c = X_c - np.outer(self.class_means[:, i], np.ones(X_c.shape[1]).T)
            S_w += D_c @ D_c.T

        return S_w


    def _compute_class_counts(self, X, y):
        """Count the number of samples in each class.

        Parameters:
        -----------
        X : np.ndarray
            Training matrix of shape ``(n_features, n_samples)``.
        y : np.ndarray
            Class-label vector of shape ``(n_samples,)``.

        Returns:
        --------
        list[int]
            Per-class sample counts in the order of `self.class_labels`.
        """
        class_counts = []
        for c in self.class_labels:
            mask = (y == c)
            class_counts.append(int(np.sum(mask)))

        return class_counts


    def _compute_between_scatter(self, d, class_counts):
        """Compute between-class scatter matrix.

        Parameters:
        -----------
        d : int
            Number of original features.
        class_counts : list[int]
            Per-class sample counts in the order of `self.class_labels`.

        Returns:
        --------
        np.ndarray
            Between-class scatter matrix of shape ``(d, d)``.
        """
        S_b = np.zeros((d, d))
        for i in range(len(self.class_labels)):
            m_c = self.class_means[:, i] - self.global_mean
            S_b += class_counts[i] * np.outer(m_c, m_c)

        return S_b


    def _compute_whitening(self, S_w, tol=1e-12):
        """Compute whitening matrix for within-class scatter.

        Some near-zero eigenvalues are removed for numerical stability.

        Parameters:
        -----------
        S_w : np.ndarray
            Within-class scatter matrix.
        tol : float
            Relative tolerance used to drop tiny eigenvalues.

        Returns:
        --------
        np.ndarray
            Whitening matrix ``P`` such that ``P @ S_w @ P.T`` is
            approximately identity on the retained subspace.
        """
        # Eigendecompose S_w.
        eigenvalues, eigenvectors = np.linalg.eigh(S_w)

        # Remove near-zero eigenvalues.
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        max_eig = float(np.max(eigenvalues)) if eigenvalues.size else 0.0
        tol_lvl = tol * max(1.0, max_eig)
        keep = eigenvalues > tol_lvl
        if not np.any(keep):
            raise ValueError(
                "Within-class scatter is numerically singular; no stable "
                "whitening directions were found."
            )

        eigenvalues = eigenvalues[keep]
        eigenvectors = eigenvectors[:, keep]

        # Build whitening matrix.
        A = np.diag(1.0 / np.sqrt(eigenvalues))
        P = A @ eigenvectors.T

        return P


    def _eigendecompose_S_b_white(self, S_b_white):
        """Find dominant discriminative directions in whitened space.

        The explained-variance ratio is stored in `self.explained_variance_ratio`.

        Parameters:
        -----------
        S_b_white : np.ndarray
            Whitened between-class scatter matrix.

        Returns:
        --------
        np.ndarray
            Matrix of top eigenvectors with shape
            ``(retained_dims, n_components)``.
        """
        # Eigendecompose whitened between-class scatter.
        eigenvalues, eigenvectors = np.linalg.eigh(S_b_white)

        # Clip tiny negative values and sort descending.
        eigenvalues = np.clip(eigenvalues, 0.0, None)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]

        # Keep top n_components directions.
        eigenvalues_clipped = eigenvalues[:self.n_components]
        V_k = eigenvectors[:, :self.n_components]

        # Compute explained variance ratios safely.
        total_variance = float(np.sum(eigenvalues))
        if total_variance <= 0.0:
            self.explained_variance_ratio = np.zeros(self.n_components)
        else:
            self.explained_variance_ratio = eigenvalues_clipped / total_variance

        return V_k
    
